# Visualization 2 - Regional Name Heatmap

This notebook creates a simple heatmap for the second mini-project question:

- Are some names more popular in certain places?
- Are names that are popular nationally also popular everywhere?
- Can we see geographic differences in naming?

The heatmap compares a few **signature names** across a few **signature departments**, plus one extra column for **France**.

In [1]:
import pandas as pd
import geopandas as gpd
import altair as alt

alt.data_transformers.disable_max_rows()
alt.renderers.enable('default')

RendererRegistry.enable('default')

In [2]:
# Load and clean the department-level baby names dataset.
names = pd.read_csv('dpt2020.csv', sep=';')
names = names[(names['preusuel'] != '_PRENOMS_RARES') & (names['dpt'] != 'XX') & (names['annais'] != 'XXXX')].copy()
names['annais'] = names['annais'].astype(int)
names['nombre'] = names['nombre'].astype(int)
names['dpt'] = names['dpt'].astype(str).str.zfill(2)

depts = gpd.read_file('departements-version-simplifiee.geojson')[['code', 'nom']]

# Focus on the recent period to keep the comparison concrete and readable.
recent = names[names['annais'].between(2010, 2020)].copy()

# A few signature names from the recent national top names.
selected_names = ['GABRIEL', 'LUCAS', 'EMMA', 'LOUIS', 'JADE', 'NATHAN', 'LOUISE', 'LÉO']

# A few signature departments with large birth volumes and different contexts.
selected_depts = ['75', '59', '69', '13', '93', '33', '44', '31']
selected_dept_labels = (
    depts[depts['code'].isin(selected_depts)]
    .assign(region_label=lambda d: d['nom'] + ' (' + d['code'] + ')')
    [['code', 'region_label']]
)

# Regional name counts and total births by department.
dept_name = (
    recent[recent['preusuel'].isin(selected_names) & recent['dpt'].isin(selected_depts)]
    .groupby(['dpt', 'preusuel'], as_index=False)['nombre']
    .sum()
)

dept_total = (
    recent[recent['dpt'].isin(selected_depts)]
    .groupby('dpt', as_index=False)['nombre']
    .sum()
    .rename(columns={'nombre': 'total_births'})
)

dept_heat = (
    dept_name.merge(dept_total, on='dpt', how='left')
    .merge(selected_dept_labels, left_on='dpt', right_on='code', how='left')
)
dept_heat['share'] = dept_heat['nombre'] / dept_heat['total_births']
dept_heat = dept_heat[['region_label', 'preusuel', 'nombre', 'total_births', 'share']]

# National reference column.
fr_name = (
    recent[recent['preusuel'].isin(selected_names)]
    .groupby('preusuel', as_index=False)['nombre']
    .sum()
)
fr_total = recent['nombre'].sum()
fr_heat = fr_name.copy()
fr_heat['region_label'] = 'France'
fr_heat['total_births'] = fr_total
fr_heat['share'] = fr_heat['nombre'] / fr_heat['total_births']
fr_heat = fr_heat[['region_label', 'preusuel', 'nombre', 'total_births', 'share']]

heatmap_data = pd.concat([fr_heat, dept_heat], ignore_index=True)

name_order = selected_names[::-1]
region_order = ['France'] + selected_dept_labels['region_label'].tolist()

heatmap_data.head()

,region_label,preusuel,nombre,total_births,share
0,France,EMMA,52515,6015739,0.008730
1,France,GABRIEL,56721,6015739,0.009429
2,France,JADE,46608,6015739,0.007748
3,France,LOUIS,50634,6015739,0.008417
4,France,LOUISE,45425,6015739,0.007551


In [3]:
chart = alt.Chart(heatmap_data).mark_rect().encode(
    x=alt.X('region_label:N', title='Region / department', sort=region_order),
    y=alt.Y('preusuel:N', title='Name', sort=name_order),
    color=alt.Color(
        'share:Q',
        title='Share of births (2010-2020)',
        scale=alt.Scale(scheme='oranges')
    ),
    tooltip=[
        alt.Tooltip('region_label:N', title='Region'),
        alt.Tooltip('preusuel:N', title='Name'),
        alt.Tooltip('nombre:Q', title='Births with this name'),
        alt.Tooltip('total_births:Q', title='Total births'),
        alt.Tooltip('share:Q', title='Share', format='.3%')
    ]
).properties(
    width=760,
    height=320,
    title='Some Baby Names Are More Regional Than Others: Departments vs France (2010-2020)'
)

labels = alt.Chart(heatmap_data).mark_text(fontSize=10).encode(
    x=alt.X('region_label:N', sort=region_order),
    y=alt.Y('preusuel:N', sort=name_order),
    text=alt.Text('share:Q', format='.2%'),
    color=alt.condition('datum.share > 0.010', alt.value('white'), alt.value('#222'))
)

chart + labels

alt.LayerChart(...)

## Why this works for Visualization 2

- It compares each department with **France** directly.
- It uses **relative popularity** instead of raw counts, so large departments do not automatically dominate.
- Darker or lighter cells immediately show where a name is more common or less common.
- This is a simple first implementation that can later evolve into an interactive filterable view.